In [ ]:
import psycopg
from collections import OrderedDict

DB_NAME = "voice_ai"
DB_USER = "vlad"
DB_PASSWORD = "Vn00193396"
DB_HOST = "10.2.4.87"
DB_PORT = "5445"

query = """
WITH linked_calls AS (
    SELECT DISTINCT
           linkedid,
           call_date
    FROM calls
    WHERE bid_id = %s
),
sorted_transcriptions AS (
    SELECT 
        t.linkedid,
        t.start,
        t.text,
        t.model,
        lc.call_date
    FROM transcribations t
    JOIN linked_calls lc ON t.linkedid = lc.linkedid
    WHERE t.text IS NOT NULL AND t.text <> ''
    ORDER BY lc.call_date, t.linkedid, t.start
)
SELECT linkedid, text, model
FROM sorted_transcriptions;
"""
conn = psycopg.connect(
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT
)
with conn.cursor() as cur:
    cur.execute(query, ('Пл2360168',))
    results = cur.fetchall()
    parts = OrderedDict()

    for linkedid, texts, model in results:
        chunk = (texts or "").strip()
        if not chunk:
            continue
        sep = " " if model else ". "
        prev = parts.get(linkedid)
        if prev is None:
            parts[linkedid] = chunk
        else:
            parts[linkedid] = f"{prev}{sep}{chunk}"

    final_text = "\n".join(
        f"Следующий диалог в разговоре: {dialog}"
        for dialog in parts.values()
    )
print(final_text)

{'linkedid =1727772346.25018987 model= 1', 'linkedid =1727773497.32223804 model= 0'}


In [ ]:
import pandas as pd
import json
import requests

filePath='./tests/BidsModels.xlsx'
df=pd.read_excel(filePath) 
srtructData=df.to_dict(orient='records')
strdict="""000000000	Не было отказа
000000005	11 Дорого диагностика
000000006	111 Техника работает норм. по тел.
000000008	12 Дорого ремонт
000000009	121 Техника направлена в стационар
000000010	13 Дорого выезд - за МКАД
000000013	151 Дубль мастера
000000014	161 Переписано  ОГД  Безнал
000000015	171 Неправильный телефон
000000016	181 Ремонт по гарантии Вендера
000000017	191 Отсутствие техники
000000019	21 Купил новый
000000021	32 Не делаем/товарная группа
000000022	33 Техника разобрана
000000023	34 Не подготовлено место
000000024	35 Антисанитарные условия
000000025	36 Не устраивает время
000000026	41 Неявка мастера
000000032	61 Отказ клиента разгов-ть по тел
000000033	71 Отсутствие клиента инф-я из ОТК
000000034	81 Нет деталей на складе
000000035	82 Не поставляется Вендером
000000036	91 Нет документации
000000046	101 Сделала др.фирма по тел.без пр"""

result_dict = {}

for line in strdict.strip().split('\n'):
    key, value = line.split('\t')
    result_dict[key] = value.strip()

print(result_dict)
for eldict in srtructData:
    print(eldict['№'])
    if eldict['№']>=1 and eldict['№']<101:
        try:
            url="https://ml.icecorp.ru:8500/refusal_check/2bclpy6R6wDboaVlmcGWkSm0GrBGCoBG/"+eldict['ЗаявкаНомер']
        
            r = requests.get(url, '', )
            print(r.text)
            data = json.loads(r.text)
            print(type(data))
            llm_reason_id = data.get("llm_reason_id")
            confidence = data.get("confidence")
            eldict['ПричинаОтказаМодели']=result_dict[llm_reason_id]
            eldict['confidence']=confidence
            eldict['IDПричиныОтказаМодели']=llm_reason_id

            print(eldict)
        except Exception as e:
            print(f"Error: str(e)")
            continue
df = pd.DataFrame(srtructData)

df.to_excel("output.xlsx")

{'000000003': '201 Дубль оператора', '000000004': 'Переписано ОГД Безнал', '000000005': '11 Дорого диагностика', '000000006': '111 Техника работает норм. по тел.', '000000007': '112 Техника работает норм.выясн.дом', '000000008': '12 Дорого ремонт', '000000009': '121 Техника направлена в стационар', '000000010': '13 Дорого выезд - за МКАД', '000000011': '131 Мастер пропал', '000000012': '141 Наш повтор (гарантия Компании)', '000000013': '151 Дубль мастера', '000000014': '161 Переписано  ОГД  Безнал', '000000015': '171 Неправильный телефон', '000000016': '181 Ремонт по гарантии Вендера', '000000017': '191 Отсутствие техники', '000000019': '21 Купил новый', '000000020': '31 Низкая квалификация мастера', '000000021': '32 Не делаем/товарная группа', '000000022': '33 Техника разобрана', '000000023': '34 Не подготовлено место', '000000024': '35 Антисанитарные условия', '000000025': '36 Не устраивает время', '000000026': '41 Неявка мастера', '000000027': '51 Отказ в оплате дорого', '000000028'